# FLX-Nano — Colab Training Runner

Thin execution wrapper for GPU training. Code lives in GitHub, compute happens here, state persists in Drive.

**Runtime → Change runtime type → GPU (A100 or L4 preferred, T4 for smoke tests only)**

In [ ]:
# Cell 1: Clone repo and install
!git clone https://github.com/YOUR_USERNAME/flux-format.git
%cd flux-format
!pip install -e '.[dev]' -q

In [ ]:
# Cell 2: Mount Google Drive for persistent .flx state
from google.colab import drive
drive.mount('/content/drive')

FLX_HUB = '/content/drive/MyDrive/flx_state'
import os
os.makedirs(FLX_HUB, exist_ok=True)
print(f'State hub: {FLX_HUB}')

In [ ]:
# Cell 3: Verify GPU
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Cell 4: Create FLX-Nano model
from flx.model import FLXNano
from flx.router import ThalamicRouter
from flx.thermal import ThermalEstimator
from flx.bridges import build_bridges
from flx.memory import MemoryController, EpisodicCompressor
from flx.meta_gen import MetaDeltaGenerator
from flx.serialization import save_flx, load_flx

model = FLXNano()
router = ThalamicRouter(d_model=512, cortex_names=model.cortex_names)
model.attach_router(router)

counts = model.count_parameters()
print('Parameter counts:')
for k, v in counts.items():
    print(f'  {k}: {v:,}')

## Phase 0 — Cortex Specialization

Forces cortices to differentiate into domain specialists.
Requires domain-labeled training data.

In [ ]:
# Cell 5: Phase 0 training
# Replace with your actual DataLoader
from flx.training.phase0_cortex import train_phase0

# history = train_phase0(
#     model, dataloader,
#     num_epochs=10, lr=1e-4,
#     device=DEVICE, log_every=100,
# )
# save_flx(model, f'{FLX_HUB}/nano_phase0.flx')
print('Phase 0: provide your DataLoader and uncomment above')

## Phase 1 — Delta-Receptive Pretraining

In [ ]:
# Cell 6: Phase 1 training
from flx.training.phase1_delta import train_phase1

# model, _, _ = load_flx(f'{FLX_HUB}/nano_phase0.flx', device=DEVICE)
# history = train_phase1(
#     model, dataloader,
#     num_epochs=20, lr=5e-5,
#     device=DEVICE, log_every=100,
# )
# save_flx(model, f'{FLX_HUB}/nano_phase1.flx')
print('Phase 1: provide your DataLoader and uncomment above')

## Phase 2 — Thermal Routing + Bridges

In [ ]:
# Cell 7: Phase 2 training
from flx.training.phase2_thermal import train_phase2

# model, _, _ = load_flx(f'{FLX_HUB}/nano_phase1.flx', device=DEVICE)
# thermal = ThermalEstimator(d_model=512)
# model.attach_thermal(thermal)
# bridges = build_bridges(model.cortex_names, d_model=512)
# model.attach_bridges(bridges)
# history = train_phase2(
#     model, dataloader,
#     num_epochs=5, lr=3e-5,
#     device=DEVICE, log_every=100,
# )
# save_flx(model, f'{FLX_HUB}/nano_phase2.flx')
print('Phase 2: provide your DataLoader and uncomment above')

## Phase 3 — Memory System

In [ ]:
# Cell 8: Phase 3 training
from flx.training.phase3_memory import train_phase3

# model, _, _ = load_flx(f'{FLX_HUB}/nano_phase2.flx', device=DEVICE)
# mem_ctrl = MemoryController(d_model=512)
# model.attach_memory(mem_ctrl)
# compressor = EpisodicCompressor(d_model=512)
# history = train_phase3(
#     model, compressor, conversation_data,
#     num_epochs=5, lr=2e-5,
#     device=DEVICE, log_every=10,
# )
# save_flx(model, f'{FLX_HUB}/nano_phase3.flx')
print('Phase 3: provide conversation_data and uncomment above')

## Phase 4 — Meta-Delta Generation

In [ ]:
# Cell 9: Phase 4 training
from flx.training.phase4_meta import train_phase4

# model, _, _ = load_flx(f'{FLX_HUB}/nano_phase3.flx', device=DEVICE)
# meta_gen = MetaDeltaGenerator(d_model=512, delta_rank=32, num_cortices=5)
# model.attach_meta_generator(meta_gen)
# history = train_phase4(
#     model, meta_gen, dataloader,
#     num_epochs=3, lr=1e-4,
#     device=DEVICE, log_every=10,
# )
# save_flx(model, f'{FLX_HUB}/nano_phase4.flx')
print('Phase 4: provide your DataLoader and uncomment above')

## Profiling (Experiment 2 — Thermal Efficiency)

In [ ]:
# Cell 10: FLOP profiling
from torch.profiler import profile, record_function, ProfilerActivity

# model, _, _ = load_flx(f'{FLX_HUB}/nano_phase2.flx', device=DEVICE)
# model.eval()
# input_ids = torch.randint(0, 32000, (1, 128), device=DEVICE)
#
# with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
#              record_shapes=True, with_flops=True) as prof:
#     with record_function('flx_forward'):
#         output = model(input_ids)
#
# print(prof.key_averages().table(sort_by='flops', row_limit=20))
print('Profiling: load a trained model and uncomment above')